# Uhifadhi wa Hoteli kwa Mwanachama wa Kipaumbele Middleware

Daftari hili linaonyesha **middleware inayotumia kazi** kwa kutumia Microsoft Agent Framework. Tunajenga zaidi kwa mfano wa mtiririko wa kazi wa masharti kwa kuongeza tabaka la middleware linalowapa wanachama wa kipaumbele haki maalum.

## Utajifunza Nini:
1. **Middleware Inayotumia Kazi**: Kukamata na kubadilisha matokeo ya kazi
2. **Ufikiaji wa Muktadha**: Kusoma na kubadilisha `context.result` baada ya utekelezaji
3. **Utekelezaji wa Mantiki ya Biashara**: Manufaa ya wanachama wa kipaumbele
4. **Kubadili Matokeo**: Kubadilisha matokeo ya kazi kulingana na hadhi ya mtumiaji
5. **Mtiririko Mmoja wa Kazi, Matokeo Tofauti**: Mabadiliko ya tabia yanayoendeshwa na middleware

## Muundo wa Mtiririko wa Kazi na Middleware:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## Tofauti Muhimu na Mtiririko wa Kazi wa Masharti:

**Bila Middleware** (14-conditional-workflow.ipynb):
- Paris haina vyumba → Elekeza kwa alternative_agent

**Kwa Middleware** (daftari hili):
- Mtumiaji wa kawaida + Paris → Hakuna vyumba → Elekeza kwa alternative_agent
- Mtumiaji wa kipaumbele + Paris → 🌟 Middleware hubadilisha! → Inapatikana → Elekeza kwa booking_agent

## Mahitaji:
- Microsoft Agent Framework imewekwa
- Uelewa wa mitiririko ya kazi ya masharti (tazama 14-conditional-workflow.ipynb)
- Tiketi ya GitHub au ufunguo wa OpenAI API
- Uelewa wa msingi wa mifumo ya middleware


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## Hatua ya 1: Tambua Mifano ya Pydantic kwa Matokeo Mbalimbali

Mifano hii inaeleza **mchoro** ambao maajenti watarudisha. Tumeongeza uwanja wa `priority_override` kufuatilia wakati middleware inabadilisha matokeo ya upatikanaji.


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Hatua ya 2: Tambua Hifadhidata ya Wanachama wa Kipaumbele

Kwa onyesho hili, tutajaribu hifadhidata ya uanachama wa kipaumbele. Kwenye uzalishaji, hii itatafuta kwenye hifadhidata halisi au API.

**Wanachama wa Kipaumbele:**
- `alice@example.com` - Mwanachama wa VIP
- `bob@example.com` - Mwanachama wa Premium  
- `priority_user` - Akaunti ya majaribio


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## Hatua ya 3: Unda Zana ya Kujisajili Hotelini

Kama vile katika mchakato wa kazi wa masharti, lakini sasa itakamatwa na middleware!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## Hatua ya 4: 🌟 Unda Middleware ya Ukaguzi wa Kipaumbele (SIFA KUU!)

Hii ni **kazi kuu** ya daftari hili. Middleware:

1. **Inakamata** wito wa kazi ya hotel_booking
2. **Inatekeleza** kazi hiyo kawaida kwa kuita `next(context)`
3. **Inachunguza** matokeo katika `context.result`
4. **Inadhulumu** matokeo ikiwa mtumiaji ni kipaumbele na hakuna vyumba vinavyopatikana
5. **Inarudisha** matokeo yaliyobadilishwa kwa wakala

**Mfumo Mkubwa:**
```python
async def my_middleware(context, next):
    await next(context)  # Tekeleza kazi
    # Sasa context.result ina matokeo ya kazi
    if some_condition:
        context.result = new_value  # Boresha!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## Hatua ya 5: Tafsiri Kazi za Masharti kwa Usanidi wa Njia

Kazi za masharti ni sawa na zile za mtiririko wa kazi wa masharti - zinachunguza matokeo yaliyo na muundo ili kubaini usanidi wa njia.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## Hatua ya 6: Tengeneza Mtendaji wa Maonyesho Maalum

Mtendaji sawa kama awali - unaonyesha matokeo ya mwisho ya mchakato wa kazi.


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## Hatua ya 7: Pakia Mabadiliko ya Mazingira

Sanidi mteja wa LLM (GitHub Models au OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Hatua ya 8: Unda Wakala wa AI kwa Middleware

**TOFAUTI KUBWA:** Tunapounda availability_agent, tunapasa kipengele cha `middleware`!

Hivi ndivyo tunavyoingiza priority_check_middleware kwenye njia ya kuitisha kazi ya wakala.


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## Hatua ya 9: Jenga Mtiririko wa Kazi

Muundo wa mtiririko wa kazi kama awali - kupelekwa kwa masharti kulingana na upatikana.


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## Hatua ya 10: Kesi ya Mtihani 1 - Mtumiaji wa Kawaida huko Paris (Hakuna Mbadala)

Mtumiaji wa kawaida anajaribu kuhifadhi Paris → Hakuna vyumba → Inapeleka kwa alternative_agent


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## Hatua 11: Kesi ya Mtihani 2 - 🌟 Mtumiaji wa Kipaumbele huko Paris (KWA Override!)

Mwanachama wa kipaumbele anajaribu kuweka Paris → Hakuna vyumba awali → 🌟 Middleware inavuka! → Inaelekeza kwa booking_agent

**Hii ni onyesho muhimu la nguvu ya middleware!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## Hatua ya 12: Kesi ya Mtihani 3 - Mtumiaji wa Kipaumbele huko Stockholm (Tayari Inapatikana)

Mtumiaji wa kipaumbele anajaribu Stockholm → Vyumba vinapatikana → Hakuna uingizwaji unaohitajika → Inaelekezwa kwa booking_agent

Hii inaonyesha kwamba middleware hufanya kazi tu inapohitajika!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## Muhimu wa Kufahamu na Dhana za Middleware

### ✅ Umejifunza Nini:

#### **1. Mfano wa Middleware Unaotegemea Kazi**

Middleware huingilia kati simu za kazi kwa kutumia kazi rahisi ya async:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # Kabla ya utekelezaji wa kazi
    print("Intercepting...")
    
    # Tekeleza kazi
    await next(context)
    
    # Baada ya utekelezaji wa kazi - chunguza matokeo
    if context.result:
        # Badilisha matokeo ikiwa yanahitajika
        context.result = modified_value
```

#### **2. Upatikanaji wa Muktadha na Kubadilisha Matokeo**

- `context.function` - Pata kazi inayoitwa
- `context.arguments` - Soma hoja za kazi
- `context.kwargs` - Pata vigezo vya ziada
- `await next(context)` - Endesha kazi
- `context.result` - Soma/badilisha matokeo ya kazi

#### **3. Utekelezaji wa Mantiki ya Biashara**

Middleware yetu hufuata manufaa ya wanachama wenye kipaumbele:
- **Watumiaji wa kawaida**: Hakuna mabadiliko, mtiririko wa kawaida
- **Watumiaji wa kipaumbele**: Badilisha "hakuna upatikanaji" → "inapatikana"
- **Mantiki ya masharti**: Hubadilisha tu inapohitajika

#### **4. Mtiririko Mmoja, Matokeo Tofauti**

Nguvu ya middleware:
- ✅ Hakuna mabadiliko kwenye muundo wa mtiririko
- ✅ Hakuna mabadiliko kwenye kazi ya chombo
- ✅ Hakuna mabadiliko kwenye mantiki ya routing inayotegemea masharti
- ✅ Middleware tu → Tabia tofauti!

### 🚀 Matumizi Halisi:

1. **Vipengele vya VIP/Premium**
   - Badilisha vizingiti vya kiwango kwa watumiaji wa premium
   - Toa upatikanaji wa kipaumbele kwa rasilimali
   - Fungua vipengele vya premium kwa njia ya mabadiliko

2. **Upimaji wa A/B**
   - Peleka watumiaji kwa utekelezaji mbalimbali
   - Jaribu vipengele vipya kwa watumiaji maalum
   - Toa vipengele hatua kwa hatua

3. **Usalama na Uzingatiaji Sheria**
   - Kagua simu za kazi
   - Zuia shughuli nyeti
   - Implemente sheria za biashara

4. **Uboreshaji wa Utendaji**
   - Hifadhi matokeo kwa watumiaji maalum
   - Ruka shughuli ghali inapowezekana
   - Ugawaji wa rasilimali kwa njia ya mabadiliko

5. **Shughulikia Makosa na Jaribu Upya**
   - Kamata na shughulikia makosa kwa busara
   - Tekeleza mantiki ya jaribio upya
   - Tumia utekelezaji mbadala kama mbadala

6. **Uandishi wa Kumbukumbu na Ufuatiliaji**
   - Fuata nyakati za utekelezaji wa kazi
   - Andika vigezo na matokeo
   - Fuata mifumo ya matumizi

### 🔑 Tofauti Muhimu na Decorators:

| Kipengele | Decorator | Middleware |
|---------|-----------|------------|
| **Uwanja** | Kazi moja | Kazi zote ndani ya agent |
| **Urahisi** | Imara wakati wa ufafanuzi | Mabadiliko wakati wa utekelezaji |
| **Muktadha** | Mdogo | Muktadha kamili wa agent |
| **Muundo** | Decorators nyingi | Mnyororo wa middleware |
| **Agent-Aware** | Hapana | Ndiyo (upatikanaji wa hali ya agent) |

### 📚 Linapofaa Kutumia Middleware:

✅ **Tumia middleware wakati:**
- Unahitaji kubadilisha tabia kulingana na hali ya mtumiaji/kikao
- Unataka kutekeleza mantiki kwa kazi nyingi
- Unahitaji upatikanaji wa muktadha wa ngazi ya agent
- Unatekeleza masuala yanayovuka mipaka (kama uandishi wa kumbukumbu, uthibitishaji, nk)

❌ **Usitumie middleware wakati:**
- Uthibitishaji rahisi wa ingizo (tumia Pydantic)
- Mantiki maalum ya kazi (kae ndani ya kazi)
- Mabadiliko ya mara moja (badilisha kazi tu)

### 🎓 Mifano Ya Juu Zaidi:

```python
# Kati ya kati nyingi (mpangilio wa utekelezaji una umuhimu!)
middleware=[
    logging_middleware,      # Rekodi kwanza
    auth_middleware,         # Kisha angalia uthibitisho
    cache_middleware,        # Kisha angalia cache
    rate_limit_middleware,   # Kisha mipaka ya kiwango
    priority_check_middleware  # Mwisho angalia kipaumbele
]

# Utekelezaji wa kati wa kati kwa masharti
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # Badilisha matokeo
    else:
        # Ruka utekelezaji kabisa
        context.result = cached_value
```

### 🔗 Dhana Zinazohusiana:

- **Agent Middleware**: Huingilia kati simu za agent.run()
- **Function Middleware**: Huingilia kati simu za kazi za zana (tunazotumia!)
- **Middleware Pipeline**: Mnyororo wa middleware unaotekelezwa kwa mpangilio
- **Uenezaji wa Muktadha**: Pitia hali kupitia mnyororo wa middleware


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
